In [53]:
import argparse
import os
import random,numpy
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [54]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.preprocessing import OneHotEncoder

In [55]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=10
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [56]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [57]:
encoder = OneHotEncoder(sparse_output=False)

for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [58]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [59]:
x_ = torch.from_numpy(train_x.to_numpy()).type('torch.FloatTensor')
y_ = torch.from_numpy(train_y.to_numpy()).type('torch.FloatTensor')
sub = torch.from_numpy(test.to_numpy()).type('torch.FloatTensor')

In [60]:
train_x = torch.utils.data.DataLoader(x_,batch_size=batch_size)
train_y = torch.utils.data.DataLoader(y_,batch_size=batch_size)
test_sub = torch.utils.data.DataLoader(sub,batch_size=batch_size)

In [61]:
def weights_init_(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight.data, gain=0.02)
        
class ENCODER(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rafire = nn.Sequential(       
            nn.Linear(57, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),
            
            nn.Linear(1524, 824),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(824, 424),
            nn.BatchNorm1d(424),
            nn.LeakyReLU(),
            
            nn.Linear(424, 824),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(824, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),

            nn.Linear(1524, 57)
            
        )

    def forward(self,x):
        
        return self.rafire(x)

encoder = ENCODER().float()
encoder= nn.DataParallel(encoder).to(device)
encoder.apply(weights_init_)
torch.save(encoder.state_dict(),f'/kaggle/working/encoder_weight.pth')

In [62]:
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight.data, gain=0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)

    elif isinstance(m, nn.GRUCell):
        nn.init.xavier_normal_(m.weight_ih.data) 
        nn.init.orthogonal_(m.weight_hh.data) 
        if m.bias is not None:
            nn.init.constant_(m.bias_ih.data, 0)  
            nn.init.constant_(m.bias_hh.data, 0)  
        
class high_regressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.rafire_1 = nn.GRUCell(57, 1524)
        self.rafire_1_b = nn.Sequential(nn.BatchNorm1d(1524),
                                        nn.LeakyReLU())
        
        self.rafire_3 = nn.GRUCell(1524, 1024)
        self.rafire_3_b = nn.Sequential(nn.BatchNorm1d(1024),
                                        nn.LeakyReLU())
        
        self.rafire_5 = nn.GRUCell(1024, 524)
        self.rafire_5_b = nn.Sequential(nn.BatchNorm1d(524),
                                        nn.LeakyReLU())

        self.rafire_7 = nn.GRUCell(524, 124)
        self.rafire_7_b = nn.Sequential(nn.BatchNorm1d(124),
                                        nn.LeakyReLU())
            
        self.rafire_8 = nn.Sequential(nn.Linear(124, 1),
                                      nn.Sigmoid())

    def forward(self,x):
        x=self.rafire_1(x)
        x=self.rafire_1_b(x)
        x=self.rafire_3(x)
        x=self.rafire_3_b(x)
        x=self.rafire_5(x)
        x=self.rafire_5_b(x)
        x=self.rafire_7(x)
        x=self.rafire_7_b(x)
        
        return self.rafire_8(x)

In [63]:
EFF_NET = high_regressor().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
EFF_NET.apply(weights_init)

DataParallel(
  (module): high_regressor(
    (rafire_1): GRUCell(57, 1524)
    (rafire_1_b): Sequential(
      (0): BatchNorm1d(1524, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (1): LeakyReLU(negative_slope=0.01)
    )
    (rafire_3): GRUCell(1524, 1024)
    (rafire_3_b): Sequential(
      (0): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (1): LeakyReLU(negative_slope=0.01)
    )
    (rafire_5): GRUCell(1024, 524)
    (rafire_5_b): Sequential(
      (0): BatchNorm1d(524, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (1): LeakyReLU(negative_slope=0.01)
    )
    (rafire_7): GRUCell(524, 124)
    (rafire_7_b): Sequential(
      (0): BatchNorm1d(124, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (1): LeakyReLU(negative_slope=0.01)
    )
    (rafire_8): Sequential(
      (0): Linear(in_features=124, out_features=1524, bias=True)
      (1): BatchNorm1d(1524, eps=1e-05, mome

In [64]:
criterion = nn.L1Loss()

optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr,betas=(beta1, 0.999))
scheduler=torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [65]:
high_rf,i,j_r,k,z_w_=100000,0,0,0,[]
    
correct,total=0,0

while(True):
    optimizer.zero_grad()
    for data,label in zip(train_x,train_y):
        #z_ = nn.Embedding(10, 3, dtype=torch.float64)
        #print(z_(torch.FloatTensor(data)))
        output = EFF_NET(encoder(data).to(device)).view(-1)
        
        err_real = criterion(output, label.to(device))
        err_real.backward()
        optimizer.step()
        #print(err_real.item())
    
    print(f"EPOCH : {i} LOSS_wl : {err_real.item()}")

    if len(z_w_)>=5:
        if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_,j_r=[],0
            print(f"lr_br:{optimizer.param_groups[0]['lr']}".upper())
            scheduler.step()
            print(f"lr_up:{optimizer.param_groups[0]['lr']}".upper())
            
    z_w_.append(err_real.item())
    if err_real.item()<high_rf:
            high_rf=err_real.item()
            torch.save(EFF_NET.state_dict(),f'/kaggle/working/{err_real.item()}.pth')
    if i==200:
        break
    i+=1

EPOCH : 0 LOSS_wl : 0.4237900674343109
EPOCH : 1 LOSS_wl : 0.6000000238418579
EPOCH : 2 LOSS_wl : 0.5
EPOCH : 3 LOSS_wl : 0.5
EPOCH : 4 LOSS_wl : 0.5
EPOCH : 5 LOSS_wl : 0.40005603432655334
EPOCH : 6 LOSS_wl : 0.4000000059604645
EPOCH : 7 LOSS_wl : 0.4000000059604645


KeyboardInterrupt: 